# Error Field Locked Mode (EFLM) Module

The purpose of this module is to determine if a locked mode will occur due to error fields during operation. An error field is any source of non-axisymmetric flux outside the plasma. The severity of this error field is quantified by the 'overlap' metric $\delta$. This overlap is compared with a threshold value computed with an empirical scaling law, and if the overlap exceeds the threshold we say a locked mode is likely to occur. A thorough description of the physics behind this module can be found in the paper [M. Pharr et al. 2024](https://arxiv.org/abs/2406.01824).


## Locking Threshold 

The threshold for locking is given by an empirical scaling law based on plasma parameters at a given time. It is not valid for all phases of the discharge, especially in regions with low current and density. As such, we only check for locking if there is a rational surface inside the plasma. The exact rule of thumb may change in the future, but for now we say $q_{90}$ < 2 is a good indicator that the plasma is in a regime where locking is possible.

In [ ]:
%load_ext autoreload
%autoreload 2

import json

import numpy as np
from omas import load_omas_h5

from popsim import PACKAGE_ROOT
from popsim.modules.tearing import ErrorFieldLocking, TearingPhase, EFUniverse, OverlapPhase, load_active_circuit_overlaps, load_tf_overlap
from popsim.simulate import SimInput, simulate, make_time_base
from popsim.interp import interp_time_dic, resolve_paths, InterpType

pcs_data = load_omas_h5(f"{PACKAGE_ROOT}/data/tearing/pcs_data_for_onsim2.h5")

scaling_laws = json.load(open(f"{PACKAGE_ROOT}/data/tearing/scalinglaws.json"))
scaling_law_terms = scaling_laws["O,L:WLS"]

popsim_time_base = make_time_base(t0=0.0, t1=20, dt=0.1)
# Trim the time base after the maximum time in the PCS data (idk why the file Ryan gave me has negative time at the end)
pcs_time_base = pcs_data['summary.time'][:20000]

ods_scaling_law_params = {
    "coeff": 10.,
    "Bt": 12.2,
    "R": 1.85,
    "ne": pcs_data['summary.line_average.n_e.value'][:20000]/1e19,
    "beta_N": pcs_data['summary.global_quantities.beta_tor_norm.value'][:20000],
    "Ip": pcs_data['summary.global_quantities.ip.value'][:20000],
    "li": pcs_data['summary.global_quantities.li.value'][:20000],
}

# For each value in the ods time base, resample the ods time trace to the popsim time base
scaling_law_params = {}
for key, value in ods_scaling_law_params.items():
    if key not in scaling_law_terms:
        continue
    if np.isscalar(value):
        scaling_law_params[key] = value
    else:
        scaling_law_params_array = np.interp(popsim_time_base, pcs_time_base, value)
        scaling_law_params[key] = {time: interpolated_value for time, interpolated_value in zip(popsim_time_base, scaling_law_params_array)}

# Hard coding to start checking at t = 4 s for now
rational_surface_exists = {}
for time in popsim_time_base:
    if time <= 4:
        rational_surface_exists[time] = 0
    else:
        rational_surface_exists[time] = 1

## Error Field Sources

Presently, the module considers two sources of error fields (with a third planned):

1. Toroidal field coils: The distribution of probable overlaps for all coils combined is given in a data file. The user can select the percentile of the distribution to use as a value in the simulation.

2. Active circuits: Coils whose currents change over the course of the shot, such as PF coils and the central solenoid. There are three components which go into calculating the overlap for the active circuits, nominal [$\delta / A$] and shifts+tilts [$\delta / A*m$]. The nominal contribution is simply multiplied by the present coil current, while the shifts and tilts are first given a randomized displacement according to engineering tolerances before being multiplied by the present coil current.

3. Static error fields (ignored): Any ferromagnetic structural material in and around the tokamak hall. Determining the exact values for these is challenging and deemed out of scope for MVP. However, the module is set up to include their effects in the future.

In [ ]:
ef_universe = EFUniverse.STARTUP_99p9 # An extremely pessimistic case

tf_overlap = load_tf_overlap(ef_universe)

error_field_locking_config = ErrorFieldLocking.Config(
    tf_overlap=tf_overlap,
    static_source_overlaps={},
)

overlaps_flattop = load_active_circuit_overlaps(OverlapPhase.FLATTOP, ef_universe)
overlaps_startup = load_active_circuit_overlaps(OverlapPhase.STARTUP, ef_universe)

ods_pf_active_circuit_current = {}
for i in range(len(pcs_data['pf_active.circuit'])):
    active_circuit_name = pcs_data[f'pf_active.circuit[{i}]']['name']
    if active_circuit_name not in overlaps_flattop.keys():
        print(f"Warning! No overlap data for {active_circuit_name}, skipping")
    else:
        ods_pf_active_circuit_current[active_circuit_name] = pcs_data[f'pf_active.circuit[{i}]']['current']['data'][:20000]

active_circuit_currents = {}
for key, value in ods_pf_active_circuit_current.items():
    active_circuit_current_array = np.interp(popsim_time_base, pcs_time_base, value)
    active_circuit_currents[key] = {time: value for time, value in zip(popsim_time_base, active_circuit_current_array)}

# Say we are in flattop if density is above 70% of the maximum value, and linearly interpolate the startup overlap up to that point
flattop_start_time = min([time for time, value in scaling_law_params["ne"].items() if value > 0.7 * max(scaling_law_params["ne"].values())])

# Linearly interpolate overlap values
overlaps_interp_tree = interp_time_dic({0: overlaps_startup, flattop_start_time: overlaps_flattop}, interp_type=InterpType.LINEAR)
active_circuit_overlaps = {time: resolve_paths(overlaps_interp_tree, time) for time in popsim_time_base}

## Set up and run the module

In [ ]:
error_field_locking_initial_state = ErrorFieldLocking.State(
    tearing_phase=TearingPhase.NONE,
)

error_field_locking_params = ErrorFieldLocking.Params(
    scaling_law_terms=scaling_law_terms,
    scaling_law_params=scaling_law_params,
    active_circuit_currents=active_circuit_currents,
    active_circuit_overlaps=active_circuit_overlaps,
    rational_surface_exists=rational_surface_exists,
)

error_field_locking_module = ErrorFieldLocking(config=error_field_locking_config)

sim_input = SimInput(time=popsim_time_base, initial_state=error_field_locking_initial_state, params=error_field_locking_params)

sim_xarray = simulate(module=error_field_locking_module, sim_inputs=sim_input)

In [ ]:
import holoviews as hv

hv.extension("bokeh")

def error_field_locking_plots(sim_xarray, aspect=3):
    parameter_plots = []
    for parameter in ["ne"]:
        parameter_plots.append(hv.Curve((sim_xarray.time, sim_xarray[f"params.scaling_law_params.{parameter}"]), vdims=parameter))
        parameter_plots[-1].opts(ylabel=parameter, aspect=aspect)

    ef_overlap_plot = hv.Curve((sim_xarray.time, np.real(sim_xarray["output.total_overlap"])), vdims="Overlap", label="EF Overlap").opts(aspect=aspect)
    threshold_plot = hv.Curve((sim_xarray.time, sim_xarray["output.locking_threshold"]), vdims="Overlap", label="Threshold").opts(aspect=aspect)
    locked_plot = hv.Scatter((sim_xarray.time, sim_xarray["output.state_dot.tearing_phase"]), vdims="Locked", label="Locked").opts(aspect=aspect)

    overlaid_status = (ef_overlap_plot.opts(color="blue") * threshold_plot.opts(color="red") * locked_plot.opts(yaxis="right", color="green")).opts(xlabel="Time [s]", multi_y=True)

    return overlaid_status + hv.Layout(parameter_plots)

all_plots = error_field_locking_plots(sim_xarray)

all_plots.cols(1)